# Routers, cascades and a budget

Three words have one meaning each in this toolkit:

- **Routing**: a `sf.Gate` decides before any answer exists, from a label of the question
  (`sf.Keywords`, `sf.LabelModel`) or from what the budget has left (`sf.budget.remaining()`).
- **Cascade**: a `sf.Gate` decides after an answer exists, from a number on that answer
  (`sf.agreement()` or a declared signal). The next stage answers the original question.
- **Orchestration**: multi-step controllers, debate and agent loops. They are not in this toolkit
  yet.

A branch that is not selected makes zero model calls, and the report says why. Each cell below
builds one recipe and shows it. Building a recipe makes no request. A recipe gives the decision
shape of a paper family only; it does not reproduce the numbers of a paper. The last cell
evaluates the recipes, and it is off by default. The README section "Choose whether a model runs"
gives the rules.

In [ ]:
import screamingface as sf

BENCHMARK = "inspect-gsm8k"
LIMIT = 3

PANEL = [
    "openrouter/openai/gpt-4o-mini",
    "openrouter/openai/gpt-4.1-mini",
    "openrouter/openai/gpt-4.1-nano",
]
SMALL = "openrouter/openai/gpt-4.1-nano"
LARGE = "openrouter/openai/gpt-4.1"
ROUTER = "openrouter/openai/gpt-4o-mini"  # the small model that gives a label
CODER = sf.Model("openrouter/openai/gpt-4.1-mini", prompt="You write and fix Python code.")
GENERAL = "openrouter/openai/gpt-4o-mini"
math_panel = sf.Fanout(PANEL) >> sf.Vote(over="last_number", normalize="numeric")

## Routing

In [ ]:
# Keywords router (binary routers): no model call to route.
topic = sf.Keywords([("math", r"(?i)\b(integral|equation|sum)\b"), ("code", r"(?i)\bpython\b")])
d1_keyword_router = sf.Gate(
    when=sf.label(topic) == "math",
    then=math_panel,
    otherwise=sf.Gate(when=sf.label(topic) == "code", then=CODER, otherwise=GENERAL),
)
d1_keyword_router

In [ ]:
# Classifier router (GraphRouter; OrgAgent DIRECT, LIGHT, FULL): a small model gives the label.
kind = sf.LabelModel(ROUTER, labels=["math", "code", "other"])
d2_label_router = sf.Gate(
    when=sf.label(kind) == "math",
    then=math_panel,
    otherwise=sf.Gate(when=sf.label(kind) == "code", then=CODER, otherwise=GENERAL),
)
d2_label_router

In [ ]:
# Router robustness: a jailbreak pattern fails the Case with fail_closed and no model call; a
# medical question goes to the large model.
jailbreak = sf.Keywords([("jailbreak", r"(?i)ignore (all|previous) instructions")])
domain = sf.LabelModel(ROUTER, labels=["medical", "other"])
d3_guarded_router = sf.Gate(
    when=sf.label(jailbreak) == "jailbreak",
    then="fail_closed",
    otherwise=sf.Gate(when=sf.label(domain) == "medical", then=LARGE, otherwise=d2_label_router),
)
d3_guarded_router

## Cascades

`escalate_if=` is the condition to call the next stage. A stage that fails or abstains escalates
too, and only the failure of the last stage fails the Case.

In [ ]:
# COPE, Frugal KG, CARGO: the large model answers only when the panel does not agree.
d4_cope_cascade = sf.Cascade([math_panel, LARGE], escalate_if=sf.agreement() < 0.8)
d4_cope_cascade

`sf.Cascade` is a short form. It returns nested `sf.Gate` values: the members of a gated stage
become optional, its panel takes `quorum=0`, and the Gate keeps the answer on the opposite
condition. The two values below are equal, and they compile to the same url4.

In [ ]:
gate_form = sf.Fanout([sf.Model(route, optional=True) for route in PANEL], quorum=0) >> sf.Gate(
    when=sf.agreement() >= 0.8,
    then=sf.Vote(over="last_number", normalize="numeric"),
    otherwise=LARGE,
)
assert gate_form == d4_cope_cascade
gate_form

In [ ]:
# STEER, per full answer: a small model's token probability decides (OpenRouter models only).
d5_steer_answer_cascade = sf.Cascade(
    [sf.Fanout([ROUTER], signals={"lp": sf.Logprob("prob")}) >> sf.Vote(), LARGE],
    escalate_if=sf.signal("lp").chosen() < 0.7,
)
d5_steer_answer_cascade

In [ ]:
# First branch to succeed: the next model runs only when the one before fails or abstains.
d6_first_success = sf.Cascade(PANEL)
d6_first_success

In [ ]:
# Adaptive sampling (ACAR): two samples; five new samples only when the two do not agree. Stage 2
# does not use the two samples again, so an escalated Case pays for 2 + 5 samples.
d10_adaptive_sampling = sf.Cascade(
    [
        sf.Fanout(sf.samples(SMALL, 2)) >> sf.Vote(over="last_number", normalize="numeric"),
        sf.Fanout(sf.samples(SMALL, 5, seeds=range(2, 7)))
        >> sf.Vote(over="last_number", normalize="numeric"),
    ],
    escalate_if=sf.agreement() < 1.0,
)
d10_adaptive_sampling

## Set-valued routing and the budget

In [ ]:
# Team of Thoughts, SMART-D: a manager label picks the experts; an expert that is not picked
# makes no call. The writer gets the answers of the experts that ran.
experts = sf.LabelModel(ROUTER, labels=["algebra", "geometry", "arithmetic"], multi=True)


def expert(topic: str) -> sf.Model:
    return sf.Model("openrouter/openai/gpt-4.1-mini", prompt=f"You are an expert in {topic}.")


d7_manager_subset = sf.Fanout(
    [
        sf.Gate(when=sf.labels(experts).has("algebra"), then=expert("algebra")),
        sf.Gate(when=sf.labels(experts).has("geometry"), then=expert("geometry")),
        sf.Gate(when=sf.labels(experts).has("arithmetic"), then=expert("arithmetic")),
    ]
) >> sf.Synthesize(LARGE, prompt="Write the final answer. The last line is: ANSWER: <number>")
d7_manager_subset

In [ ]:
# Willingness to pay (OmniRouter, RouterBench): sf.budget.remaining() needs a sf.Budgeted root.
d8_budget_choice = sf.Budgeted(
    sf.Gate(when=sf.budget.remaining() >= 0.02, then=LARGE, otherwise=SMALL), usd=0.05
)
# A cost cap per Case (MTRouter, VineLM) around the d4 cascade.
d9_budget_cap = sf.Budgeted(d4_cope_cascade, usd=0.05)
d8_budget_choice, d9_budget_cap

## Evaluate the recipes

⚠️ This cell calls models and costs money 💵. Each recipe gets a name, so the report can tell
them apart, and the "Branches" section of each Case shows the branch that each Gate took and the
branches that did not run. `report.candidates[name].url4` is the compiled recipe.

**`inspect-gsm8k` needs an Engine with the `inspect` extra** (see
`12_inspect_evals_benchmarks`).

In [ ]:
RUN_EVALUATION = False

recipes = {
    "d1-keyword-router": d1_keyword_router,
    "d2-label-router": d2_label_router,
    "d3-guarded-router": d3_guarded_router,
    "d4-cope-cascade": d4_cope_cascade,
    "d5-steer-answer-cascade": d5_steer_answer_cascade,
    "d6-first-success": d6_first_success,
    "d7-manager-subset": d7_manager_subset,
    "d8-budget-choice": d8_budget_choice,
    "d9-budget-cap": d9_budget_cap,
    "d10-adaptive-sampling": d10_adaptive_sampling,
}


def named(name: str, recipe: sf.Recipe) -> sf.Recipe:
    # A sf.Budgeted stays at the root, so the name goes on the recipe inside it.
    if isinstance(recipe, sf.Budgeted):
        return sf.Budgeted(
            sf.Pipeline([recipe.recipe], name=name), usd=recipe.usd, tokens=recipe.tokens
        )
    return sf.Pipeline([recipe], name=name)


candidates = [named(name, recipe) for name, recipe in recipes.items()]
report = (
    sf.evaluate(candidates, benchmark=BENCHMARK, limit=LIMIT, progress=True)
    if RUN_EVALUATION
    else None
)
report